# EDA – goodbooks-10k

Basic overview of the raw data: columns, data types, missing values, counts and data quality issues.

Data is downloaded by `python src/ingest.py` into `data/raw/`.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 50)
RAW = "../data/raw"

books = pd.read_csv(f"{RAW}/books.csv")
ratings = pd.read_csv(f"{RAW}/ratings.csv")
to_read = pd.read_csv(f"{RAW}/to_read.csv")
tags = pd.read_csv(f"{RAW}/tags.csv")
book_tags = pd.read_csv(f"{RAW}/book_tags.csv")

dfs = {"books": books, "ratings": ratings, "to_read": to_read, "tags": tags, "book_tags": book_tags}

## Overview of files

In [2]:
pd.DataFrame({
    name: {"rows": len(df), "cols": df.shape[1], "nan_total": df.isna().sum().sum(), "duplicate_rows": df.duplicated().sum()}
    for name, df in dfs.items()
}).T

,rows,cols,nan_total,duplicate_rows
books,10000,23,2975,0
ratings,5976479,3,0,0
to_read,912705,2,0,0
tags,34252,2,0,0
book_tags,999912,3,0,6


In [3]:
def column_summary(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nan": df.isna().sum(),
        "nan_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(),
    })

## books.csv

In [4]:
books.head()

,book_id,goodreads_book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,title,language_code,average_rating,ratings_count,work_ratings_count,work_text_reviews_count,ratings_1,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,"The Hunger Games (The Hunger Games, #1)",eng,4.34,4780653,4942365,155254,66715,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...
1,2,3,3,4640799,491,439554934,9.780440e+12,"J.K. Rowling, Mary GrandPré",1997.0,Harry Potter and the Philosopher's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,eng,4.44,4602479,4800065,75867,75504,101676,455024,1156318,3011543,https://images.gr-assets.com/books/1474154022m...,https://images.gr-assets.com/books/1474154022s...
2,3,41865,41865,3212258,226,316015849,9.780316e+12,Stephenie Meyer,2005.0,Twilight,"Twilight (Twilight, #1)",en-US,3.57,3866839,3916824,95009,456191,436802,793319,875073,1355439,https://images.gr-assets.com/books/1361039443m...,https://images.gr-assets.com/books/1361039443s...
3,4,2657,2657,3275794,487,61120081,9.780061e+12,Harper Lee,1960.0,To Kill a Mockingbird,To Kill a Mockingbird,eng,4.25,3198671,3340896,72586,60427,117415,446835,1001952,1714267,https://images.gr-assets.com/books/1361975680m...,https://images.gr-assets.com/books/1361975680s...
4,5,4671,4671,245494,1356,743273567,9.780743e+12,F. Scott Fitzgerald,1925.0,The Great Gatsby,The Great Gatsby,eng,3.89,2683664,2773745,51992,86236,197621,606158,936012,947718,https://images.gr-assets.com/books/1490528560m...,https://images.gr-assets.com/books/1490528560s...


In [5]:
column_summary(books)

,dtype,nan,nan_%,unique
book_id,int64,0,0.00,10000
goodreads_book_id,int64,0,0.00,10000
best_book_id,int64,0,0.00,10000
work_id,int64,0,0.00,10000
books_count,int64,0,0.00,597
isbn,object,700,7.00,9300
isbn13,float64,585,5.85,9153
authors,object,0,0.00,4664
original_publication_year,float64,21,0.21,293
original_title,object,585,5.85,9274


In [6]:
books.language_code.value_counts(dropna=False).head(10)

language_code
eng      6341
en-US    2070
NaN      1084
en-GB     257
ara        64
en-CA      58
fre        25
ind        21
spa        20
ger        13
Name: count, dtype: int64

In [7]:
# publication year range – negative years are ancient texts
books.original_publication_year.describe()

count    9979.000000
mean     1981.987674
std       152.576665
min     -1750.000000
25%      1990.000000
50%      2004.000000
75%      2011.000000
max      2017.000000
Name: original_publication_year, dtype: float64

In [8]:
# isbn13 is stored as float -> needs conversion to string in ETL
books[["isbn", "isbn13"]].head()

,isbn,isbn13
0,439023483,9.780439e+12
1,439554934,9.780440e+12
2,316015849,9.780316e+12
3,61120081,9.780061e+12
4,743273567,9.780743e+12


In [9]:
# isbn has lost leading zeros (ISBN-10 must have 10 chars) -> zfill(10) in ETL
books.isbn.dropna().str.len().value_counts()

isbn
9     5573
10    2699
8      916
7      112
Name: count, dtype: int64

In [10]:
# books without a cover use a 'nophoto' placeholder
books.image_url.str.contains("nophoto").sum()

3332

## ratings.csv

In [11]:
ratings.head()

,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [12]:
column_summary(ratings)

,dtype,nan,nan_%,unique
user_id,int64,0,0.0,53424
book_id,int64,0,0.0,10000
rating,int64,0,0.0,5


In [13]:
ratings.rating.value_counts().sort_index()

rating
1     124195
2     359257
3    1370916
4    2139018
5    1983093
Name: count, dtype: int64

In [14]:
n_users, n_books = ratings.user_id.nunique(), ratings.book_id.nunique()
print(f"users: {n_users:,}, books: {n_books:,}")
print("duplicate (user, book) pairs:", ratings.duplicated(["user_id", "book_id"]).sum())
print(f"density: {len(ratings) / (n_users * n_books):.2%}")

users: 53,424, books: 10,000


duplicate (user, book) pairs: 0
density: 1.12%


In [15]:
pd.DataFrame({
    "ratings_per_user": ratings.groupby("user_id").size().describe(),
    "ratings_per_book": ratings.groupby("book_id").size().describe(),
}).round(1)

,ratings_per_user,ratings_per_book
count,53424.0,10000.0
mean,111.9,597.6
std,26.1,1267.3
min,19.0,8.0
25%,96.0,155.0
50%,111.0,248.0
75%,128.0,503.0
max,200.0,22806.0


### books ↔ ratings integrity

In [16]:
# every rating must point to an existing book, every book must have at least one rating
print("ratings with unknown book_id:", (~ratings.book_id.isin(books.book_id)).sum())
print("books without any rating:   ", (~books.book_id.isin(ratings.book_id)).sum())
print("min ratings per book:", ratings.groupby("book_id").size().min())
print("min ratings per user:", ratings.groupby("user_id").size().min())

ratings with unknown book_id: 0
books without any rating:    0
min ratings per book: 8


min ratings per user: 19


## to_read.csv

In [17]:
to_read.head()

,user_id,book_id
0,9,8
1,15,398
2,15,275
3,37,7173
4,34,380


In [18]:
column_summary(to_read)

,dtype,nan,nan_%,unique
user_id,int64,0,0.0,48871
book_id,int64,0,0.0,9986


## tags.csv and book_tags.csv

In [19]:
column_summary(tags)

,dtype,nan,nan_%,unique
tag_id,int64,0,0.0,34252
tag_name,object,0,0.0,34252


In [20]:
column_summary(book_tags)

,dtype,nan,nan_%,unique
goodreads_book_id,int64,0,0.0,10000
tag_id,int64,0,0.0,34252
count,int64,0,0.0,9401


In [21]:
print("duplicate rows:", book_tags.duplicated().sum())
print("negative counts:", (book_tags["count"] < 0).sum())
print("book_tags uses goodreads_book_id (not book_id), all present in books:",
      book_tags.goodreads_book_id.isin(books.goodreads_book_id).all())

duplicate rows: 6
negative counts: 6
book_tags uses goodreads_book_id (not book_id), all present in books: True


In [22]:
# most used tags – many are not genres (to-read, owned, favorites ...)
(book_tags.groupby("tag_id")["count"].sum()
    .nlargest(25)
    .rename(tags.set_index("tag_id").tag_name)
    .to_frame("total_count"))

,total_count
tag_id,
to-read,140718761
currently-reading,7507958
favorites,4503173
fiction,3688819
fantasy,3548157
young-adult,1848306
classics,1756920
books-i-own,1317235
romance,1231926


## Findings

- **ratings / to_read** are clean: no NaN, no duplicates, `book_id` 1–10000 matches `books.csv`.
- Integrity holds both ways: every rating has its book and every book has at least 8 ratings.
- 6M ratings, 53k users, 10k books, density ~1.1 %. Ratings are skewed positive (4 and 5 ≈ 69 %).
- Each user has 19–200 ratings (looks capped at 200); books have 8–22806 ratings (strong popularity bias).

**To handle in ETL:**
1. `books`: NaN in `isbn` (7 %), `isbn13`, `original_title`, `language_code` (11 %).
2. `isbn` lost leading zeros (pad to 10 chars); `isbn13` stored as float (convert to string).
3. `language_code`: merge `eng`, `en-US`, `en-GB`, `en-CA` into `en`.
4. ~1/3 of books have a placeholder cover.
5. `book_tags`: 6 duplicate rows and 6 negative counts; uses `goodreads_book_id` instead of `book_id`.
6. Tags: keep only genre-like tags, drop shelf tags (`to-read`, `owned`, `favorites` …).